# Lesson03. 机器狗出发！让它指哪走哪

**教学主题：** 学习控制机器人的精准移动。

**核心目标：** 理解简单的二维坐标概念。用代码控制机器人前进、后退与转向。

**课程安排：**

- **前20分钟（概念课）：** 介绍高级运动指令 `move(x, y, yaw)`。用地面上的十字胶带做比喻，理解x（前后）、y（左右）、yaw（旋转）三个方向。

- **后100分钟（路径挑战）：**
  - **直线冲刺：** 编写脚本，让Go2向前走2米后停下。
  - **完美转身：** 让Go2原地旋转90度、180度。
  - **综合任务【挑战】：** 编写程序，让Go2在地板上走一个正方形路径。

## 3.1 导入依赖并初始化客户端

和Lesson02一样，先导入SDK并建立与机器狗的连接。

In [ ]:
import time  # 时间模块，用于控制延时
import sys   # 系统模块

# 导入宇树SDK通信和运动控制模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道
ChannelFactoryInitialize(0, "ens37")

# 创建并初始化运动控制客户端
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

### 3.1.1 向前移动

调用 `Move(vx, vy, vyaw)` 方法控制机器狗运动：
- `vx`：前后方向速度（正值前进，负值后退），单位 m/s
- `vy`：左右方向速度（正值向左，负值向右），单位 m/s
- `vyaw`：旋转角速度（正值逆时针，负值顺时针），单位 rad/s

In [ ]:
# 向前移动：vx=0.3 m/s，vy=0，vyaw=0（只向前，不侧移，不转弯）
ret = sport_client.Move(0.3, 0, 0)
print("返回值: ", ret)  # 返回0表示成功

### 3.1.2 向左移动

设置 `vy` 为正值，机器狗将向左侧移。

In [ ]:
# 向左侧移：vx=0，vy=0.3 m/s，vyaw=0
sport_client.Move(0, 0.3, 0)

### 3.1.3 原地转弯

设置 `vyaw` 为正值，机器狗将逆时针旋转。

In [ ]:
# 原地转弯：vx=0，vy=0，vyaw=0.5 rad/s（逆时针旋转）
sport_client.Move(0, 0, 0.5)

## 3.2 挑战：走正方形路径

综合运用前进和侧移指令，让机器狗在地面上走出一个正方形。

**思路：** 依次沿四个方向移动相同的距离——先向前，再向左，再向后，最后向右，即可形成一个正方形路径。

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def move_continuous(v_x, v_y, duration, interval=0.4):
    """
    持续发送运动指令，使机器人按指定速度运动指定时长。
    
    由于单次Move指令的有效时间约为0.5秒，需要循环发送指令来实现持续运动。
    
    Args:
        v_x (float): x方向速度（前后），单位 m/s
        v_y (float): y方向速度（左右），单位 m/s
        duration (float): 总运动时长（秒）
        interval (float): 发送指令的时间间隔（秒），需小于单次指令有效时长0.5秒
    """
    start_time = time.time()
    while time.time() - start_time < duration:
        sport_client.Move(v_x, v_y, 0)
        time.sleep(interval)
    # 发送零速度指令，停止运动
    sport_client.Move(0, 0, 0)
    time.sleep(0.5)  # 等待停止指令生效


def move_square(base_vx=0.3, base_vy=0.3, scale=1.0, base_time=2.0):
    """
    控制机器人走正方形路径。
    
    通过依次沿x正方向、y正方向、x负方向、y负方向移动，
    形成一个闭合的正方形轨迹。
    
    Args:
        base_vx (float): x方向基础速度（m/s）
        base_vy (float): y方向基础速度（m/s）
        scale (float): 正方形边长比例因子，值越大边长越长
        base_time (float): 基础运动时间（秒），与scale相乘得到每条边的实际运动时间
    """
    # 计算每条边的运动时长
    edge_duration = base_time * scale
    
    # 第一条边：向前移动（x正方向）
    move_continuous(base_vx, 0, edge_duration)
    
    # 第二条边：向左移动（y正方向）
    move_continuous(0, base_vy, edge_duration)
    
    # 第三条边：向后移动（x负方向）
    move_continuous(-base_vx, 0, edge_duration)
    
    # 第四条边：向右移动（y负方向）
    move_continuous(0, -base_vy, edge_duration)


if __name__ == "__main__":
    # 设置正方形大小的比例因子，scale越大，正方形边长越长
    square_scale = 1.0
    move_square(scale=square_scale)